# Numerické metody — cvičební notebook

Každý úkol říká: **jaká metoda**, **jaká data**, **co chceš získat**.  
Doplň kód do prázdné buňky, spusť, zkontroluj výsledek.

---

In [ ]:
import sys, os, math

BASE = os.path.abspath('.')

sys.path.insert(0, os.path.join(BASE, '01_nelinearni_rovnice'))
sys.path.insert(0, os.path.join(BASE, '02_polynomy_a_interpolace'))
sys.path.insert(0, os.path.join(BASE, '03_soustavy_linearnich_rovnic'))
sys.path.insert(0, os.path.join(BASE, '04_aproximace'))
sys.path.insert(0, os.path.join(BASE, '06_numericka_integrace'))
sys.path.insert(0, os.path.join(BASE, '07_diferencialni_rovnice'))

from bisekce          import bisection
from regula_falsi_A   import regula_falsi
from newtonroot       import newton
from steffansen_A     import steffensen
from fixed_point_A    import fixed_point_iteration
from halley_A         import halley
from newton_horner_A  import newton_horner   # koefs od nejnižší mocniny
from lin_interpolace_po_castech_A import linear_interpolation
from gauss            import gauss
from gaussseidel      import gauss_seidel
from jacobi           import jacobi
from mnc              import lsa
from simpson          import simpson_rule
from romberg_A        import romberg_quadrature
from euler            import euler_step
from runge_kutta_A    import runge_kutta_4

print('Importy OK')

---
## 1. Nelineární rovnice
### 1.1 Bisekce

**Kdy:** máš interval [a,b] kde f(a)·f(b) < 0, chceš spolehlivou (ale pomalou) metodu.  
**Konvergence:** lineární — každá iterace zmenší interval na polovinu.

**Úkol:**  
Najdi kořen funkce `f(x) = x³ - x - 2` na intervalu `[1, 2]`.  
Použij toleranci `1e-8`, max iterací `100`.  
Ověř: `f(koren) ≈ 0`.

In [ ]:
# --- Bisekce ---


import math

def bisection(f, a, b, tol, max_iter):
    """
    Numerické hledání kořene funkce pomocí metody půlení intervalu.
    
    :param f: Funkce, jejíž kořen hledáme
    :param a: Levá mez intervalu
    :param b: Pravá mez intervalu
    :param tol: Tolerance (přesnost)
    :param max_iter: Maximální počet iterací
    :return: Kořen funkce nebo None, pokud metoda selže
    """
    fa = f(a)
    fb = f(b)

    # Kontrola, zda v intervalu vůbec může být kořen (změna znaménka)
    if fa * fb > 0:
        print("Error: No sign change, can't guarantee a root.")
        return None

    c = a
    for i in range(max_iter):
        c = (a + b) / 2
        fc = f(c)

        # Kontrola, zda jsme dosáhli dostatečné přesnosti
        if abs(fc) < tol or (b - a) / 2 < tol:
            return c

        # Rozhodnutí, kterou polovinu intervalu si ponecháme
        if fa * fc < 0:
            b = c
            fb = fc
        else:
            a = c
            fa = fc

    print("Error: Did not converge.")
    return None

# Příklad použití:
# root = bisection(lambda x: x**2 - 4, 0, 5, 1e-6, 100)
# print(f"Kořen je: {root}")


function_for_bis = lambda x: x**3 - x - 2

# TODO: zavolej bisection(f, a, b, tol, max_iter)

koren = bisection(function_for_bis,1,2,1e-6,100)


if koren != None:
    print(f"Funkce má kořen(polovinu) v bodě {koren:4f}")
else:
    print(f"Funkce nemá kořen, f(a)*f(b) nikdy není menší jak nula a nedochází ke změně znaménka")
















Funkce má kořen(polovinu) v bodě 1.521380


In [10]:
# ŘEŠENÍ
koren = bisection(f, 1, 2, 1e-8, 100)
print(f'koren = {koren:.10f}')         # ≈ 1.5213797068
print(f'f(koren) = {f(koren):.2e}')

NameError: name 'f' is not defined

### 1.2 Regula Falsi

**Kdy:** stejné podmínky jako bisekce, ale nový bod je průsečík sečny → rychlejší u nelineárních f.  
**Nevýhoda:** jeden konec intervalu může zůstat zafixovaný.

**Úkol:**  
Najdi kořen `f(x) = e^x - 3x` na intervalu `[1, 2]`.  
Tolerance `1e-9`, max `200` iterací.

In [18]:



import math

def regula_falsi(f, a, b, tol, max_iter):
    """
    Hledání kořene rovnice f(x)=0 metodou regula falsi (sečnová metoda s intervalem).
    Rozdíl od bisekce: nový bod c je průsečík sečny, ne střed intervalu.

    :param f: Funkce, jejíž kořen hledáme
    :param a: Levá mez intervalu
    :param b: Pravá mez intervalu
    :param tol: Tolerance (přesnost)
    :param max_iter: Maximální počet iterací
    :return: Odhad kořene nebo None při selhání metody
    """
    fa = f(a)
    fb = f(b)

    if fa * fb > 0:
        print("Error: No sign change, can't guarantee a root.")
        return None

    c = a
    for i in range(max_iter):
        if abs(fb - fa) < 1e-12:
            print("Error: Zero denominator (f(a) == f(b)).")
            return None

        c = a - fa * (b - a) / (fb - fa)
        fc = f(c)

        if math.isnan(fc) or math.isinf(fc):
            print("Error: Did not converge (NaN/Inf).")
            return None

        if abs(fc) < tol:
            return c

        if fa * fc < 0:
            b = c
            fb = fc
        else:
            a = c
            fa = fc

    print("Error: Did not converge within max iterations.")
    return None




# --- Regula Falsi ---
fun = lambda x: math.exp(x) - 3*x

vysledek = regula_falsi(fun,1,2, 1e-9, 100)

print(vysledek)








1.5121345512034226


In [ ]:
# ŘEŠENÍ
koren = regula_falsi(f, 1, 2, 1e-9, 200)
print(f'koren = {koren:.10f}')    # ≈ 1.5121345516
print(f'f(koren) = {f(koren):.2e}')

### 1.3 Newton-Raphson

**Kdy:** znáš derivaci, máš dobrý počáteční odhad. Kvadratická konvergence.  
**Riziko:** diverguje pokud f'(x) ≈ 0 nebo špatný x0.

**Úkol:**  
Najdi kořen `f(x) = cos(x) - x` (slavná rovnice, kořen se nazývá Dottie number).  
Derivace: `f'(x) = -sin(x) - 1`.  
Počáteční odhad `x0 = 0.5`, tolerance `1e-12`, max `50` iterací.

In [ ]:
# --- Newton-Raphson ---
# Definuj f(x) = cos(x) - x a její derivaci fd(x)
# Zavolej newton(f, fd, x0, tol, max_iter) s x0=0.5, tol=1e-12, max_iter=50

koren = None
print(f'koren = {koren}')

In [ ]:
# ŘEŠENÍ
f  = lambda x: math.cos(x) - x
fd = lambda x: -math.sin(x) - 1
koren = newton(f, fd, 0.5, 1e-12, 50)
print(f'koren = {koren:.12f}')   # ≈ 0.739085133215
print(f'f(koren) = {f(koren):.2e}')

### 1.4 Steffensenova metoda

**Kdy:** chceš rychlost Newtona ale **bez derivace** — odhaduje ji z hodnot funkce.  
**Konvergence:** kvadratická (jako Newton).

**Úkol:**  
Najdi kořen `f(x) = x² - 5` (tj. √5).  
Počáteční odhad `x0 = 2.0`, tolerance `1e-10`, max `100` iterací.  
Ověř: `koren² ≈ 5`.

In [ ]:
# --- Steffensen ---
# Definuj f(x) = x² - 5
# Zavolej steffensen(f, x0, tol, max_iter) s x0=2.0, tol=1e-10, max_iter=100

koren = None
print(f'koren = {koren}')

In [ ]:
# ŘEŠENÍ
f = lambda x: x**2 - 5
koren = steffensen(f, 2.0, 1e-10, 100)
print(f'koren = {koren:.12f}')   # ≈ 2.236067977500
print(f'koren² = {koren**2:.10f}')

### 1.5 Metoda pevného bodu

**Kdy:** přepíšeš `f(x)=0` na tvar `x = φ(x)` a φ je kontrakce (|φ'(x)| < 1 v okolí kořene).  
**Konvergence:** lineární.

**Úkol:**  
Rovnice: `x³ - x - 1 = 0` → přepíšeme na `x = (x+1)^(1/3)` → `φ(x) = (x+1)^(1/3)`.  
Počáteční odhad `x0 = 1.5`, tolerance `1e-8`, max `200` iterací.

In [ ]:
# --- Fixed Point ---
# Rovnice x³ - x - 1 = 0 → přepiš na tvar x = φ(x)
# Zavolej fixed_point_iteration(phi, x0, tol, max_iter) s x0=1.5, tol=1e-8, max_iter=200

koren = None
print(f'koren = {koren}')

In [ ]:
# ŘEŠENÍ
phi = lambda x: (x + 1)**(1/3)
koren = fixed_point_iteration(phi, 1.5, 1e-8, 200)
print(f'koren = {koren:.10f}')    # ≈ 1.3247179572
print(f'f(koren) = {koren**3 - koren - 1:.2e}')

### 1.6 Halleyho metoda

**Kdy:** máš i 2. derivaci. Konvergence kubická (rychlejší než Newton).  
**Cena:** musíš počítat f, f', f'' v každém kroku.

**Úkol:**  
Najdi kořen `f(x) = x³ - 2`.  
`f'(x) = 3x²`, `f''(x) = 6x`.  
Počáteční odhad `x0 = 1.5`, tolerance `1e-12`, max `30` iterací.  
Výsledek je ∛2.

In [ ]:
# --- Halley ---
# Definuj f(x) = x³ - 2, f'(x) = 3x², f''(x) = 6x
# Zavolej halley(f, f1, f2, x0, tol, max_iter) s x0=1.5, tol=1e-12, max_iter=30

koren = None
print(f'koren = {koren}')

In [ ]:
# ŘEŠENÍ
f   = lambda x: x**3 - 2
f1  = lambda x: 3*x**2
f2  = lambda x: 6*x
koren = halley(f, f1, f2, 1.5, 1e-12, 30)
print(f'koren = {koren:.12f}')   # ≈ 1.259921049895
print(f'koren³ = {koren**3:.12f}')

### 1.7 Newton-Horner (kořen polynomu)

**Kdy:** hledáš kořen polynomu. Hornerovo schéma efektivně počítá P(x) a P'(x).  
**Koeficienty:** od nejnižší mocniny → `[a0, a1, a2, ...]` znamená `a0 + a1*x + a2*x² + ...`

**Úkol:**  
Polynom: `P(x) = -6 + 11x - 6x² + x³`  
Koeficienty (od nejnižší): `[-6, 11, -6, 1]`  
Kořeny tohoto polynomu jsou 1, 2, 3. Najdi ten kolem `x0 = 2.5`.  
Tolerance `1e-10`, max `100` iterací.

In [ ]:
# --- Newton-Horner ---
# Polynom: P(x) = -6 + 11x - 6x² + x³  (kořeny: 1, 2, 3)
# Definuj koefs od nejnižší mocniny a najdi kořen kolem x0=2.5
# Zavolej newton_horner(coefs, x0, tol, max_iter), tol=1e-10, max_iter=100

koren = None
print(f'koren = {koren}')

In [ ]:
# ŘEŠENÍ
coefs = [-6, 11, -6, 1]   # P(x) = x³ - 6x² + 11x - 6
koren = newton_horner(coefs, 2.5, 1e-10, 100)
print(f'koren = {koren:.10f}')   # ≈ 3.0
# Zkus i x0=0.5 → najde kořen 1, x0=1.5 → najde kořen 2

---
## 2. Interpolace
### 2.1 Lineární interpolace po částech

**Kdy:** máš tabulku naměřených hodnot a chceš odhadnout hodnotu mezi body.  
**Princip:** na každém intervalu [x_i, x_{i+1}] je přímka.

**Úkol:**  
Teplota vzduchu v průběhu dne (každé 4 hodiny):

| Čas (h) | Teplota (°C) |
|---------|-------------|
| 0       | 8.0         |
| 4       | 6.5         |
| 8       | 11.0        |
| 12      | 18.5        |
| 16      | 20.0        |
| 20      | 15.0        |
| 24      | 10.0        |

Odhadni teplotu v čase `t = 10` hod, `t = 14` hod a `t = 22` hod.

In [ ]:
# --- Lineární interpolace ---
# Data jsou ze zadání výše — přepiš je
# Zavolej linear_interpolation(x_vals, y_vals, t) pro t = 10, 14, 22

t10 = None
t14 = None
t22 = None
print(f't=10h → {t10} °C')
print(f't=14h → {t14} °C')
print(f't=22h → {t22} °C')

In [ ]:
# ŘEŠENÍ
x_vals = [0, 4, 8, 12, 16, 20, 24]
y_vals = [8.0, 6.5, 11.0, 18.5, 20.0, 15.0, 10.0]
t10 = linear_interpolation(x_vals, y_vals, 10)
t14 = linear_interpolation(x_vals, y_vals, 14)
t22 = linear_interpolation(x_vals, y_vals, 22)
print(f't=10h → {t10:.3f} °C')    # ≈ 14.75
print(f't=14h → {t14:.3f} °C')    # ≈ 19.25
print(f't=22h → {t22:.3f} °C')    # ≈ 12.5

---
## 3. Soustavy lineárních rovnic
### 3.1 Gaussova eliminace

**Kdy:** přímá metoda, vždy najde řešení (pokud existuje). O(n³).  
**Pozor:** základní verze bez pivotace může selhat na nulový pivot.

**Úkol:**  
Vyřeš soustavu:
```
 2x +  y -  z =  8
-3x - y + 2z = -11
-2x + y + 2z = -3
```
Správné řešení: `x=2, y=3, z=-1`.

In [ ]:
# --- Gaussova eliminace ---
# Soustava:  2x +  y -  z =  8
#           -3x -  y + 2z = -11
#           -2x +  y + 2z = -3
# Definuj matici A a vektor b, zavolej gauss(A, b)

x = None
print(f'x = {x}')   # má být [2.0, 3.0, -1.0]

In [ ]:
# ŘEŠENÍ
A = [
    [ 2,  1, -1],
    [-3, -1,  2],
    [-2,  1,  2]
]
b = [8, -11, -3]
x = gauss(A, b)
print(f'x = {[round(v, 6) for v in x]}')

### 3.2 Gauss-Seidelova metoda

**Kdy:** iterativní metoda, vhodná pro velké řídké soustavy.  
**Podmínka konvergence:** diagonálně dominantní matice (|a_ii| > Σ|a_ij| pro j≠i).  
**Rozdíl od Jacobiho:** nové hodnoty x[i] se používají **ihned** v téže iteraci → rychlejší.

**Úkol:**  
Soustava (diagonálně dominantní):
```
10x +  y + 2z = 44
 x + 10y + 3z = 51
 2x +  3y + 10z = 61
```
Max `500` iterací, tolerance `1e-8`.  
Správné řešení: `x=1, y=2, z=4`.

In [ ]:
# --- Gauss-Seidel ---
# Soustava: 10x +  y + 2z = 44
#            x + 10y + 3z = 51
#           2x +  3y + 10z = 61
# Definuj A a b, zavolej gauss_seidel(A, b, max_iter, tol), max_iter=500, tol=1e-8

x = None
print(f'x = {x}')   # má být [1.0, 2.0, 4.0]

In [ ]:
# ŘEŠENÍ
A = [
    [10, 1, 2],
    [1, 10, 3],
    [2,  3, 10]
]
b = [44, 51, 61]
x = gauss_seidel(A, b, 500, 1e-8)
print(f'x = {[round(v, 6) for v in x]}')

### 3.3 Jacobiho metoda

**Kdy:** stejné jako Gauss-Seidel, ale nové hodnoty se **nepoužívají ihned** — aktualizuje se až po celé iteraci.  
**Výhoda:** snadno paralelizovatelná. **Nevýhoda:** pomalejší konvergence.

**Úkol:**  
Stejná soustava jako výše. Porovnej počet iterací s Gauss-Seidelem.  
Max `1000` iterací, tolerance `1e-8`.

In [ ]:
# --- Jacobi ---
# Stejná soustava jako Gauss-Seidel výše
# Zavolej jacobi(A, b, max_iter, tol), max_iter=1000, tol=1e-8

x = None
print(f'x = {x}')   # má být [1.0, 2.0, 4.0]

In [ ]:
# ŘEŠENÍ
A = [
    [10, 1, 2],
    [1, 10, 3],
    [2,  3, 10]
]
b = [44, 51, 61]
x = jacobi(A, b, 1000, 1e-8)
print(f'x = {[round(v, 6) for v in x]}')

---
## 4. Aproximace — metoda nejmenších čtverců (LSA)

**Kdy:** máš zašuměná naměřená data a chceš proložit polynomem stupně (n-1).  
**Princip:** minimalizuje součet čtverců odchylek Σ(y_i - P(x_i))².

**Úkol:**  
Naměřená data (průhyb nosníku v závislosti na zatížení):

| Zatížení (kN) | Průhyb (mm) |
|--------------|-------------|
| 1            | 2.1         |
| 2            | 4.0         |
| 3            | 6.2         |
| 4            | 7.9         |
| 5            | 10.1        |
| 6            | 12.0        |

Prolož **lineárním polynomem** (n=2, tj. `a0 + a1*x`).  
Pak zkus **kvadratický** (n=3). Porovnej koeficienty.

In [ ]:
# --- LSA ---
# Data jsou ze zadání výše — přepiš je
# Zavolej lsa(x_data, y_data, n) pro n=2 (lineární) a n=3 (kvadratický)

koefs_lin  = None
koefs_quad = None
print(f'Lineární:    {koefs_lin}')
print(f'Kvadratický: {koefs_quad}')

In [ ]:
# ŘEŠENÍ
x_data = [1, 2, 3, 4, 5, 6]
y_data = [2.1, 4.0, 6.2, 7.9, 10.1, 12.0]
koefs_lin  = lsa(x_data, y_data, 2)
koefs_quad = lsa(x_data, y_data, 3)
print(f'Lineární:    a0={koefs_lin[0]:.4f}, a1={koefs_lin[1]:.4f}')
x_pred = 7
y_lin  = koefs_lin[0] + koefs_lin[1]*x_pred
y_quad = koefs_quad[0] + koefs_quad[1]*x_pred + koefs_quad[2]*x_pred**2
print(f'Předpověď x=7: lineární={y_lin:.3f} mm, kvadratický={y_quad:.3f} mm')

---
## 6. Numerická integrace
### 6.1 Simpsonovo pravidlo

**Kdy:** přesná integrace hladkých funkcí. Na každých dvou intervalech fituje parabolu.  
**Podmínka:** `n` musí být **sudé** a ≥ 2.  
**Chyba:** O(h⁴) → mnohem přesnější než lichoběžník (O(h²)).

**Úkol A:**  
Spočítej `∫₀^π sin(x) dx`. Přesná hodnota = 2.  
Použij `n=10` dělení. Jaká je absolutní chyba?

**Úkol B:**  
Spočítej `∫₁^e (1/x) dx`. Přesná hodnota = 1 (protože ln(e)-ln(1)=1).  
Použij `n=100` dělení.

In [ ]:
# --- Simpson ---
# Úkol A: spočítej ∫₀^π sin(x) dx  (přesná hodnota = 2), n=10
# Úkol B: spočítej ∫₁^e (1/x) dx   (přesná hodnota = 1), n=100
# Zavolej simpson_rule(f, a, b, n)

I_A = None
I_B = None
print(f'A: I = {I_A}')
print(f'B: I = {I_B}')

In [ ]:
# ŘEŠENÍ
I_A = simpson_rule(math.sin, 0, math.pi, 10)
print(f'A: I = {I_A:.10f}, chyba = {abs(I_A - 2):.2e}')

I_B = simpson_rule(lambda x: 1/x, 1, math.e, 100)
print(f'B: I = {I_B:.10f}, chyba = {abs(I_B - 1):.2e}')

### 6.2 Rombergova metoda

**Kdy:** chceš vysokou přesnost bez ručního nastavování `n`.  
**Princip:** Richardson extrapolace na lichoběžníkové pravidlo — iterativně zdvojuje body a eliminuje chybu.  
**Výhoda:** automaticky konverguje na zadanou toleranci.

**Úkol:**  
Spočítej `∫₀^1 e^(-x²) dx` (Gaussův integrál — nemá analytické vyjádření).  
Referenční hodnota: `≈ 0.7468241328`.  
Tolerance `1e-10`, max `20` iterací.  

Pak zkus stejný integrál Simpsonem s `n=1000` — kdo je přesnější?

In [ ]:
# --- Romberg ---
# Spočítej ∫₀^1 e^(-x²) dx  (referenční hodnota: 0.7468241328124271)
# Zavolej romberg_quadrature(f, a, b, tol, max_iter), tol=1e-10, max_iter=20
# Pak stejný integrál Simpsonem s n=1000 — porovnej přesnost

I_romberg = None
I_simpson = None
print(f'Romberg: {I_romberg}')
print(f'Simpson: {I_simpson}')

In [ ]:
# ŘEŠENÍ
f = lambda x: math.exp(-x**2)
ref = 0.7468241328124271
I_romberg = romberg_quadrature(f, 0, 1, 1e-10, 20)
I_simpson = simpson_rule(f, 0, 1, 1000)
print(f'Romberg:  {I_romberg:.12f}, chyba = {abs(I_romberg - ref):.2e}')
print(f'Simpson:  {I_simpson:.12f}, chyba = {abs(I_simpson - ref):.2e}')

---
## 7. Diferenciální rovnice
### 7.1 Eulerova metoda

**Kdy:** nejjednodušší řešič ODE. Vhodný pro hrubý odhad nebo výukové účely.  
**Princip:** `y_{n+1} = y_n + h·f(x_n, y_n)`  
**Chyba:** O(h) — globální chyba roste lineárně s krokem h.

**Úkol:**  
ODE: `y' = y`, počáteční podmínka `y(0) = 1`.  
Analytické řešení: `y(x) = e^x`.  
Spočítej numericky na intervalu `[0, 1]` s krokem `h=0.1` (n=10 kroků).  
Porovnej `y(1)` numericky vs. `e¹ = 2.71828...`

In [ ]:
# --- Euler ---
# ODE: y' = y,  y(0) = 1  → analytické řešení y(x) = e^x
# Definuj f(x,y), x0, y0, h=0.1, n=10
# Zavolej euler_step(f, x0, y0, h, n)  → vrací seznam y[0]..y[n]

ys = None
print(f'Euler y(1) = {ys}')

In [ ]:
# ŘEŠENÍ
f  = lambda x, y: y
ys = euler_step(f, 0, 1, 0.1, 10)
print(f'Euler y(1)      = {ys[-1]:.8f}')   # ≈ 2.59374246
print(f'Přesná y(1)=e¹  = {math.e:.8f}')
print(f'Chyba           = {abs(ys[-1] - math.e):.4f}')   # ≈ 0.1245

### 7.2 Runge-Kutta 4. řádu (RK4)

**Kdy:** standardní metoda pro ODE. Zlatý standard přesnosti vs. výpočetní náklady.  
**Princip:** 4 evaluace f per krok, chyba O(h⁴) — mnohem přesnější než Euler.  
**Vrací:** seznam dvojic `[(x0, y0), (x1, y1), ...]`

**Úkol A:**  
Stejná ODE jako Euler: `y' = y`, `y(0) = 1`, h=0.1, n=10.  
Porovnej chybu RK4 vs. Euler pro `y(1)`.

**Úkol B:**  
ODE: `y' = -2xy`, `y(0) = 1`.  
Analytické řešení: `y(x) = e^(-x²)`.  
h=0.1, n=20. Porovnej `y(2)` numericky vs. přesně.

In [ ]:
# --- RK4 ---
# Úkol A: y' = y,  y(0) = 1,  h=0.1, n=10  → porovnej y(1) s e¹
# Úkol B: y' = -2xy,  y(0) = 1,  h=0.1, n=20  → porovnej y(2) s e^(-4)
# Zavolej runge_kutta_4(f, x0, y0, h, n)  → vrací [(x0,y0), (x1,y1), ...]

pts_A = None
pts_B = None
print(f'A: y(1) = {pts_A}')
print(f'B: y(2) = {pts_B}')

In [ ]:
# ŘEŠENÍ
pts_A = runge_kutta_4(lambda x, y: y, 0, 1, 0.1, 10)
y_rk4  = pts_A[-1][1]
print(f'A: RK4 y(1) = {y_rk4:.12f}')    # ≈ 2.718281828
print(f'A: Chyba RK4 = {abs(y_rk4 - math.e):.2e}')

pts_B = runge_kutta_4(lambda x, y: -2*x*y, 0, 1, 0.1, 20)
y_rk4_B  = pts_B[-1][1]
y_exac_B = math.exp(-4)
print(f'B: RK4 y(2)  = {y_rk4_B:.12f}')
print(f'B: Přesně    = {y_exac_B:.12f}')
print(f'B: Chyba RK4 = {abs(y_rk4_B - y_exac_B):.2e}')

---
## Shrnutí — Kdy co použít

| Problém | Metoda | Výhoda | Nevýhoda |
|---------|--------|--------|----------|
| Kořen f(x)=0, máš interval | Bisekce | Vždy konverguje | Pomalá (lineární) |
| Kořen f(x)=0, máš interval | Regula Falsi | Rychlejší než bisekce | Jeden konec fixní |
| Kořen f(x)=0, máš f' | Newton | Kvadratická konvergence | Nutná derivace, může divergovat |
| Kořen f(x)=0, bez f' | Steffensen | Kvadratická, bez derivace | Složitější implementace |
| Kořen f(x)=0, máš φ(x) | Fixed point | Jednoduchý | Konverguje jen pokud \|φ'\|<1 |
| Kořen f(x)=0, máš f,f',f'' | Halley | Kubická konvergence | Nutné 2 derivace |
| Kořen polynomu | Newton-Horner | Efektivní (Horner) | Jen pro polynomy |
| Hodnota mezi body | Lin. interpolace | Rychlá | Nespojitá derivace |
| Soustava Ax=b (přímá) | Gauss | Exaktní, vždy | O(n³) |
| Soustava Ax=b (iterační) | Gauss-Seidel | Rychlejší než Jacobi | Vyžaduje diag. dominanci |
| Soustava Ax=b (iterační) | Jacobi | Paralelizovatelný | Pomalejší než G-S |
| Proložení dat | LSA | Robustní na šum | Stupeň polynomu ad hoc |
| Integrál, hladká f | Simpson | Přesné, O(h⁴) | Sudý n |
| Integrál, vysoká přesnost | Romberg | Auto-adaptivní | Složitější |
| ODE, hrubý odhad | Euler | Nejjednodušší | Velká chyba O(h) |
| ODE, standard | RK4 | Přesné O(h⁴) | 4× víc evaluací f |